In [33]:
from google_play_scraper import reviews, Sort
import pandas as pd
import numpy as np
from datetime import datetime

## Sample Test Data

In [34]:
# chatgpt
result, continuation_token = reviews(
    "com.openai.chatgpt",
    lang="en",
    country="us",
    sort = Sort.NEWEST,
    count=10
)

df_chatgpt = pd.DataFrame(result)
df_chatgpt.insert(0, "app_name", "ChatGPT")
df_chatgpt



,app_name,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,ChatGPT,f8fb3975-7c94-402a-8520-4b29dacc451c,Erin H (Littleredpanda),https://play-lh.googleusercontent.com/a-/ALV-U...,uninstall so we have water for our kids. wake up,1,0,1.2026.202,2026-07-29 22:45:23,None,None,1.2026.202
1,ChatGPT,98b53b59-7ff6-495d-91a0-2b144237edd0,Orukpe Peculiar,https://play-lh.googleusercontent.com/a/ACg8oc...,I love 💕 this app I want to,5,1,1.2025.140,2026-07-29 19:45:26,None,None,1.2025.140
2,ChatGPT,7a139aa5-85ec-49e6-9a27-3b53b32f9198,Dilip Yadav Yadav,https://play-lh.googleusercontent.com/a/ACg8oc...,don't give answer fast,1,0,1.2026.202,2026-07-29 18:52:17,None,None,1.2026.202
3,ChatGPT,3788de94-93fb-4232-87da-6aca12f7ba31,Sarwarhaider Haiderkonain,https://play-lh.googleusercontent.com/a/ACg8oc...,ai aap. open. ai wonderful,5,1,1.2026.202,2026-07-29 18:03:29,None,None,1.2026.202
4,ChatGPT,11725073-1019-4173-817e-74e4ab305ad2,Ryan Fitzpatrick,https://play-lh.googleusercontent.com/a-/ALV-U...,make my nan cum,5,0,1.2026.195,2026-07-29 11:05:34,None,None,1.2026.195
5,ChatGPT,59e00806-efcb-44f7-90b6-470ec2318dc5,Matthew,https://play-lh.googleusercontent.com/a-/ALV-U...,Thanks for rolling back my god that was a horr...,5,0,1.2026.195,2026-07-29 06:00:08,None,None,1.2026.195
6,ChatGPT,bf01e85b-cb3b-4f8e-8676-a67ca40670e6,Rigsel Dorjee,https://play-lh.googleusercontent.com/a/ACg8oc...,fix the damn authentication error,1,10,1.2026.195,2026-07-25 19:37:54,None,None,1.2026.195
7,ChatGPT,4e4654b5-0355-47a2-bcb5-68d5b1592aae,Boy Bom,https://play-lh.googleusercontent.com/a/ACg8oc...,"""AI promised it could analyze my uploaded vide...",1,2,1.2026.188,2026-07-24 21:03:48,None,None,1.2026.188
8,ChatGPT,f72758cc-da38-4c99-bc05-985f41273b05,Asha Malla,https://play-lh.googleusercontent.com/a-/ALV-U...,Gjdu,5,2,None,2026-07-19 11:42:32,None,None,None
9,ChatGPT,45e62226-bc8a-4279-aa71-be1547a6db9d,nancy sarpong,https://play-lh.googleusercontent.com/a/ACg8oc...,the time limit is my problem,2,4,1.2026.188,2026-07-18 06:48:13,None,None,1.2026.188


## Connect to MySQl

In [35]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [36]:
mysql_password = os.getenv("MYSQL_PASSWORD")

mysql_host = os.getenv("MYSQL_HOST")
mysql_user = os.getenv("MYSQL_USER")
mysql_database = os.getenv("MYSQL_DATABASE")

In [37]:
print(mysql_host)
print(mysql_user)

localhost
root


In [38]:
import mysql.connector

connection = mysql.connector.connect(
    host=mysql_host,
    user=mysql_user,
    password=mysql_password,
    database=mysql_database
)
cursor = connection.cursor()

## Ingestion

In [39]:
# insert into apps
def insert_app(cursor, df_chatgpt):
    app_name = df_chatgpt["app_name"].iloc[0]
    category = "productivity"
    sql = """
    INSERT IGNORE INTO Apps
    (app_name,
    category)

    VALUES
    (%s,%s)
    """
    
    cursor.execute(sql,(app_name,category))
    real_app_id = cursor.lastrowid
    return real_app_id



In [40]:
# insert into app versions
def insert_version(cursor, df_chatgpt, real_app_id):
    versions=df_chatgpt["reviewCreatedVersion"].unique()
    version_mapping = {}

    for v in versions:
        cursor.execute("""
        INSERT INTO AppVersions
        (app_id,version)
        VALUES(%s,%s)
        """,(real_app_id,v))
    
        real_version_id = cursor.lastrowid
        version_mapping[v] = real_version_id

    return version_mapping



In [41]:
# insert into ingestion runs
def create_ingestion_run(cursor,df_chatgpt):
    
    sql = """
    INSERT INTO IngestionRuns
    (collection_time, volume, source, app_list, language, country,
    filter_method, target_review_count, run_status, error_count, start_time, end_time)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    cursor.execute(sql, (
        datetime.now(),
        len(df_chatgpt),
        "Google Play",
        'Chatgpt',
        'English',
        'US',
        'NEWEST',
        200,
        "Success" if len(df_chatgpt) == 200 else "Fail",
        200-len(df_chatgpt),
        datetime.now(),
        datetime.now()
    ))
    real_run_id = cursor.lastrowid
    return real_run_id


In [42]:
# check duplication 

def duplicate_exists(
    cursor,
    review_id
):
    sql = """
    SELECT COUNT(*)
    FROM RawReviews
    WHERE review_id=%s
    """
    cursor.execute(sql, (review_id,))
    count = cursor.fetchone()[0]
    return count > 0

In [43]:
# check usefulness
def get_quality_flags(text):
    flags = []
    if len(text) == 0:
        flags.append("Missing Review")
    if len(text.split()) <= 2:
        flags.append("Low Signal")
    if not text or str(text).strip() == "":
        flags.append("No Actual Words")
    return flags

def get_flag_id(flag_name):
    mapping = {
        "Missing Review": 1,
        "Low Signal": 2,
        "No Actual Words": 3
    }
    return mapping[flag_name]

In [44]:
# insert into raw reviews

def insert_raw_reviews(cursor, df_chatgpt,real_run_id, real_app_id, version_mapping):
    duplicate_count = 0
    inserted_count = 0
    raw_id_list = [] 
    
    sql = """
    INSERT INTO RawReviews
    (review_id,
        run_id,
        app_id,
        version_id,
        user_name,
        user_image,
        content,
        score,
        thumbsUpCount,
        ReviewCreatedVersion,
        at,
        replyContent,
        RepliedAt)
    VALUES
    (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """
    
    for _, row in df_chatgpt.iterrows():
        review_id = row["reviewId"]
        if duplicate_exists(cursor, review_id):
            duplicate_count += 1
            continue

        version_real = row["reviewCreatedVersion"]
        if pd.isna(version_real):
            real_version_id = None
        else:
            real_version_id = version_mapping[version_real]

        cursor.execute(sql,
            (review_id,
                real_run_id,                         
                real_app_id,                         
                real_version_id,                          
                row["userName"],
                row["userImage"],
                row["content"],
                float(row["score"]),
                int(row["thumbsUpCount"]),
                row["reviewCreatedVersion"],
                row["at"],
                row["replyContent"],
                row["repliedAt"]
            ))
        
        inserted_count += 1
        raw_id_list.append(cursor.lastrowid)
        
    return inserted_count, duplicate_count, raw_id_list


In [45]:
# insert into processed reviews
def insert_processed_reviews(cursor, df, raw_id_list, real_app_id, version_mapping):
    processed_count = 0
    total_flags = 0
    
    sql_processed = """
    INSERT INTO ProcessedReviews
    (raw_id,
        app_id,
        version_id,
        text,
        rating,
        date)
    VALUES
    (%s,%s,%s,%s,%s,%s)
    """
    
    sql_flags = """
    INSERT INTO QualityMapping (processed_id, flag_id)
    VALUES (%s, %s)
    """
    
    for idx, (_, row) in enumerate(df_chatgpt.iterrows()):
        real_raw_id = raw_id_list[idx]

        version_real = row["reviewCreatedVersion"]
        if pd.isna(version_real):
            real_version_id = None
        else:
            real_version_id = version_mapping[version_real]
        
        cursor.execute(
            sql_processed,
            (real_raw_id,
                real_app_id,
                real_version_id,
                row["content"],
                float(row["score"]),
                row["at"]))
        real_processed_id = cursor.lastrowid

        review_flags = get_quality_flags(row["content"])
        for flag in review_flags:
            flag_id = get_flag_id(flag)
            cursor.execute(sql_flags,(real_processed_id, flag_id))
            total_flags += 1
        
        processed_count += 1

    return processed_count, total_flags

In [46]:
# commit
def final(conn):
    conn.commit()
    print("Database committed successfully.")


In [47]:
#############################################
# Main Pipeline
#############################################

def main(): 
    df = df_chatgpt 
    app_id = insert_app(cursor, df) 
    version_mapping = insert_version(cursor, df, app_id) 
    run_id = create_ingestion_run(cursor, df)
    inserted, duplicates, raw_id_list = insert_raw_reviews(cursor, df,run_id, app_id, version_mapping) 
    processed, total_flags_inserted = insert_processed_reviews(cursor, df, raw_id, app_id, version_mapping) 
     
    
    final(conn) 

    cursor.close() 
    conn.close() 
    print("\nPipeline Finished Successfully.") 

    print(f"Number of Input Reviews: {len(df)}")
    print(f"Number of Raw Inserted: {inserted}")
    print(f"Number of Duplicates Skipped: {duplicates}")
    print(f"Number of Processed Reviews: {processed}")
    print(f"Total Quality Flags: {total_flags_inserted}")
    

main()


DatabaseError: 1205 (HY000): Lock wait timeout exceeded; try restarting transaction